In [1]:
def make_query(params):
    import requests

    response = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers={
        "User-Agent": "CategoryCollector (ingenuity.wikipedia@gmail.com)"
    })
    return response.json()

In [ ]:
def categoryinfo(pages):
    page_dict = {}
    
    for i in range(0, len(pages), 50):
        results = make_query({
            "format": "json",
            "action": "query",
            "prop": "categoryinfo",
            "titles": "|".join(pages[i:i+50])
        })["query"]["pages"]
        print(results)
    
        for page in results:
            page_dict[results[page]["title"]] = page

    return page_dict

In [35]:
import csv

wikiproject_names = {}

with open("../data/wikiprojects.csv", "r") as f:
	reader = csv.reader(f)
	for line in reader:
		wikiproject_names[line[0]] = (line[1], line[2])

In [69]:
api_url = "https://api.wp1.openzim.org/v1/projects"

import requests
session = requests.Session()

def get_random_pages_in_project(project_name, quality, count=5000):
	import math, random

	gathered_articles = 0
	pagination_data = session.get(f"{api_url}/{project_name}/articles?quality={quality}")
	num_articles = pagination_data.json()["pagination"]["total"]
	num_pages = math.ceil(num_articles / 500)
	shuffled_pages = random.sample([(i+1) for i in range(num_pages)], num_pages)
	
	while gathered_articles < count:
		pagenum = shuffled_pages.pop()
		data = session.get(f"{api_url}/{project_name}/articles?page={pagenum}&numRows=500&quality={quality}")
		data = data.json()
		for article in data["articles"]:
			yield article["article"]
		gathered_articles += len(data["articles"])

In [74]:
import os

if not os.path.isfile("../data/random_articles.csv"):
	with open("../data/random_articles.csv", "a") as f:
		writer = csv.writer(f)

		for wp, (wpname, quality) in wikiproject_names.items():
			for article in get_random_pages_in_project(wpname.replace(" ", "_"), quality):
				writer.writerow((wp, article))

In [85]:
def get_lead(pages):
	page_leads = {}
	for i in range(0, len(pages), 50):
		sliced_pages = pages[i:i+50]
		results = make_query({
			"format": "json",
			"action": "query",
			"titles": "|".join(sliced_pages),
			"prop": "extracts",
			"explaintext": "1",
			"exintro": "1"
		})["query"]["pages"]

		for key in results:
			page_leads[results[key]["title"]] = results[key]["extract"] if "extract" in results[key] else ""
	return page_leads

In [78]:
get_lead(["Elizabeth II", "Charles III"])

{'Charles III': "Charles III (Charles Philip Arthur George; born 14 November 1948) is King of the United Kingdom and 14 other Commonwealth realms.\nCharles was born during the reign of his maternal grandfather, King George VI, and became heir apparent when his mother, Queen Elizabeth II, acceded to the throne in 1952. He was created Prince of Wales in 1958 and his investiture was held in 1969. Charles was educated at Cheam School and Gordonstoun, and later spent six months at the Timbertop campus of Geelong Grammar School in Victoria, Australia. After completing a history degree at the University of Cambridge, he served in the Royal Air Force and the Royal Navy from 1971 to 1976. He married Lady Diana Spencer in 1981 and they had two sons, William and Harry. Charles and Diana divorced in 1996 after years of estrangement and well-publicised extramarital affairs. Diana died the following year from injuries sustained in a car crash. In 2005, Charles married his long-time partner, Camilla 

In [87]:
def get_all_lead_data():
	import time

	with open("../data/lead_data.csv", "a") as f1:
		writer = csv.writer(f1)
		with open("../data/random_articles.csv", "r") as f2:
			reader = csv.reader(f2)
			current_chunk = []
			to_write = []
			for line in reader:
				current_chunk.append(line)

				if len(current_chunk) >= 50:
					try:
						leads = get_lead([page[1] for page in current_chunk])
					except Exception as e:
						print("Error:", e)
						leads = {}
						time.sleep(10)
					current_chunk = []
					for page in leads:
						if len(leads[page]) < 50:
							continue
						shortened_lead = " ".join(leads[page].split(" ")[:100])
						shortened_lead = shortened_lead.replace("\n", " ")
						to_write.append((page, shortened_lead))
					writer.writerows(to_write)
					to_write = []

get_all_lead_data()

In [94]:
def get_pages_content(pages):
	page_dict = {}
	
	for i in range(0, len(pages), 50):
		results = make_query({
			"action": "query",
			"prop": "revisions",
			"titles": "|".join(pages[i:i+50]),
			"rvslots": "*",
			"rvprop": "content",
			"format": "json",
			"formatversion": 2
		})
		
		for item in results["query"]["pages"]:
			if "missing" in item:
				continue
			page_dict[item["title"]] = item["revisions"][0]["slots"]["main"]["content"]
	
	return page_dict

In [100]:
import re

def get_projects_from_content(content):
	wp_regex = r"{{(WikiProject [^|}]+?)\s*[|}][^\n]+?}}"
	projects = re.findall(wp_regex, content)
	try: projects.remove("WikiProject banner shell")
	except: pass
	return projects

def get_projects_for_articles():
	with open("../data/all_data.csv", "a") as f1:
		writer = csv.writer(f1)
		with open("../data/lead_data.csv", "r") as f2:
			reader = csv.reader(f2)
			current_chunk = []

			for line in reader:
				current_chunk.append(line)
				all_data = {f"Talk:{page[0]}": page for page in current_chunk}

				if len(current_chunk) >= 50:
					try:
						content = get_pages_content(list(all_data.keys()))
					except Exception as e:
						print("Error:", e)
						content = {}
					new_data = []
					for page in content:
						new_data.append((
							all_data[page][0], all_data[page][1],
							"|".join(get_projects_from_content(content[page]))
						))
					writer.writerows(new_data)
					current_chunk = []

get_projects_for_articles()